In [ ]:
# Enable CUDA GPU - SELECT GPU T4 ×2 ON KAGGLE!
import os
import sys

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("🚀 CUDA GPU Setup...")
    !pip uninstall -y cupy-cuda11x cupy-cuda12x cupy -q 2>/dev/null || true
    !pip install -q cupy-cuda11x
    
    try:
        import cupy as cp
        device = cp.cuda.Device()
        gpu_name = cp.cuda.runtime.getDeviceProperties(device.id)['name'].decode('utf-8')
        total_mem, free_mem = cp.cuda.Device().mem_info
        print(f"✓ GPU: {gpu_name}, Memory: {total_mem / 1e9:.1f} GB")
        USE_GPU = True
    except Exception as e:
        print(f"⚠️ GPU setup failed: {e}")
        USE_GPU = False
else:
    print("Local CPU mode")
    USE_GPU = False

print("="*60)

In [ ]:
# Clone Repository
import os, sys

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    if not os.path.exists('/kaggle/working/NLP_PROJECT_2025'):
        !git clone -q -b mohab https://github.com/MohabYasser2/NLP_PROJECT_2025.git /kaggle/working/NLP_PROJECT_2025
    else:
        !cd /kaggle/working/NLP_PROJECT_2025 && git fetch -q origin mohab && git reset --hard origin/mohab
    
    # Clear old cache
    !rm -f /kaggle/working/*_processed.pkl
    sys.path.insert(0, '/kaggle/working/NLP_PROJECT_2025')
    print("✓ Repo ready")
else:
    sys.path.insert(0, os.path.abspath('..'))

In [ ]:
# Detect Dataset Paths
import glob

if IS_KAGGLE:
    data_files = glob.glob('/kaggle/input/**/*.txt', recursive=True)
    if data_files:
        dataset_dir = os.path.dirname(data_files[0])
        TRAIN_FILE = os.path.join(dataset_dir, 'train.txt')
        DEV_FILE = os.path.join(dataset_dir, 'val.txt') if 'val.txt' in str(data_files) else os.path.join(dataset_dir, 'dev.txt')
        TEST_FILE = os.path.join(dataset_dir, 'test.txt')
    else:
        print("⚠️ No dataset found! Add your dataset in Kaggle.")
else:
    TRAIN_FILE = '../data/train.txt'
    DEV_FILE = '../data/val.txt'
    TEST_FILE = '../data/test.txt'

OUTPUT_FILE = '/kaggle/working/submission.csv' if IS_KAGGLE else 'submission.csv'
print(f"✓ Train: {TRAIN_FILE}")

In [ ]:
# MEMORY-OPTIMIZED TRAINING FOR GPU T4 x2 (30GB RAM LIMIT)
import pickle
import numpy as np
from pathlib import Path
from src.preprocessing import prepare_dataset
from src.models.logreg_model import LogisticRegressionModel

MODEL_PATH = '/kaggle/working/NLP_PROJECT_2025/models/logreg_model.pkl' if IS_KAGGLE else '../models/logreg_model.pkl'

if os.path.exists(MODEL_PATH):
    print("="*70)
    print("📦 LOADING PRE-TRAINED MODEL")
    print("="*70)
    
    with open(MODEL_PATH, 'rb') as f:
        model = pickle.load(f)
    
    # Load dev data
    cache_dir = Path('/kaggle/working') if IS_KAGGLE else Path('../data')
    cache_file = cache_dir / 'val_processed.pkl'
    
    if cache_file.exists():
        with open(cache_file, 'rb') as f:
            data = pickle.load(f)
        dev_texts, dev_labels = data['texts'], data['labels']
    else:
        dev_texts, dev_labels = prepare_dataset(str(DEV_FILE), str(cache_file.with_suffix('')))
    
    # Evaluate
    metrics = model.evaluate(dev_texts, dev_labels, window_size=7)
    print(f"\n✓ Accuracy: {metrics['accuracy']*100:.2f}%")
    print(f"✓ DER: {metrics['der']*100:.2f}%")
    print("="*70)
    
else:
    print("="*70)
    print("🚀 ULTRA-OPTIMIZED TRAINING (GPU T4 x2, <30GB RAM)")
    print("="*70)
    print("\n📋 AGGRESSIVE Memory Configuration:")
    print("   • Features: 3,000 (MINIMAL for <30GB)")
    print("   • N-grams: 1-2 (character bigrams only)")
    print("   • Vocab: 10k sample (not 20k!)")
    print("   • Window: 5 (smaller context)")
    print("   • STREAMING: No matrix storage, direct GPU training")
    print("   • Batch: 1024, LR: 0.1, Iter: 100")
    print("\n⏱️  Expected: ~10-15 min, Peak RAM: ~12-18GB")
    print("="*70)
    
    # Load data with chunking
    cache_dir = Path('/kaggle/working') if IS_KAGGLE else Path('../data')
    
    print("\n[1/5] Loading training data...")
    train_cache = cache_dir / 'train_processed.pkl'
    if train_cache.exists():
        with open(train_cache, 'rb') as f:
            data = pickle.load(f)
        train_texts, train_labels = data['texts'], data['labels']
    else:
        train_texts, train_labels = prepare_dataset(str(TRAIN_FILE), str(train_cache.with_suffix('')))
    
    print(f"✓ Loaded {len(train_texts)} training samples")
    
    print("\n[2/5] Loading dev data...")
    dev_cache = cache_dir / 'val_processed.pkl'
    if dev_cache.exists():
        with open(dev_cache, 'rb') as f:
            data = pickle.load(f)
        dev_texts, dev_labels = data['texts'], data['labels']
    else:
        dev_texts, dev_labels = prepare_dataset(str(DEV_FILE), str(dev_cache.with_suffix('')))
    
    print(f"✓ Loaded {len(dev_texts)} dev samples")
    
    # ULTRA-AGGRESSIVE MEMORY OPTIMIZATION
    print("\n[3/5] Building ultra-compact model...")
    model = LogisticRegressionModel(
        learning_rate=0.1,  # Higher LR for faster convergence with fewer features
        max_iter=100,  # Reduced iterations
        regularization=1e-3,
        max_features=3000,  # DRASTICALLY REDUCED: 3k instead of 5k
        ngram_range=(1, 2),  # MINIMAL: Only 1-2 grams (not 1-3!)
        batch_size=1024  # Larger batches for GPU
    )
    
    window_size = 5  # Smaller window (was 7)
    
    # CRITICAL: Use only 10k training samples for vocabulary
    print("\n[4/5] Building vocabulary from 10k sample...")
    vocab_contexts = []
    sample_count = 0
    for text, label_seq in zip(train_texts, train_labels):
        if sample_count >= 10000:
            break
        min_len = min(len(text), len(label_seq))
        for j in range(min_len):
            start = max(0, j - window_size // 2)
            end = min(len(text), j + window_size // 2 + 1)
            vocab_contexts.append(text[start:end])
            sample_count += 1
            if sample_count >= 10000:
                break
    
    model.vectorizer.fit(vocab_contexts)
    print(f"✓ Vocabulary: {len(model.vectorizer.vocabulary):,} features from {len(vocab_contexts):,} samples")
    del vocab_contexts  # Free memory immediately
    
    # STREAMING TRANSFORM + TRAIN (never store full matrix!)
    print("\n[5/5] Streaming training (zero matrix storage)...")
    print("🚀 Processing data in mini-batches directly to GPU...")
    
    # Initialize weights manually
    import gc
    num_features = len(model.vectorizer.vocabulary)
    num_classes = 15
    
    try:
        import cupy as cp
        GPU_ACTIVE = True
        model.weights = cp.random.randn(num_features, num_classes).astype(cp.float32) * 0.01
        model.bias = cp.zeros(num_classes, dtype=cp.float32)
        xp = cp
        print(f"✓ Model on GPU: {num_features} features × {num_classes} classes")
    except:
        GPU_ACTIVE = False
        model.weights = np.random.randn(num_features, num_classes).astype(np.float32) * 0.01
        model.bias = np.zeros(num_classes, dtype=np.float32)
        xp = np
        print(f"✓ Model on CPU: {num_features} features × {num_classes} classes")
    
    # STREAM TRAINING: Process in tiny chunks to avoid RAM spikes
    chunk_size = 500  # Process 500 sentences at a time (smaller!)
    transform_batch = 10000  # Transform 10k contexts at a time
    
    print(f"\nTraining on {len(train_texts)} sentences ({model.max_iter} epochs)...")
    
    from tqdm import tqdm
    total_chunks = (len(train_texts) + chunk_size - 1) // chunk_size
    
    for epoch in range(model.max_iter):
        epoch_loss = 0
        batch_count = 0
        
        # Progress bar for this epoch
        progress = tqdm(range(0, len(train_texts), chunk_size), 
                       desc=f"Epoch {epoch+1}/{model.max_iter}", 
                       total=total_chunks,
                       unit="chunk")
        
        for i in progress:
            # Extract contexts for tiny chunk
            chunk_contexts = []
            chunk_labels = []
            
            for text, label_seq in zip(train_texts[i:i+chunk_size], 
                                      train_labels[i:i+chunk_size]):
                min_len = min(len(text), len(label_seq))
                for j in range(min_len):
                    start = max(0, j - window_size // 2)
                    end = min(len(text), j + window_size // 2 + 1)
                    chunk_contexts.append(text[start:end])
                    chunk_labels.append(label_seq[j])
            
            if len(chunk_contexts) == 0:
                continue
            
            # Transform in sub-batches (SILENT mode)
            X_chunks = []
            for k in range(0, len(chunk_contexts), transform_batch):
                sub_contexts = chunk_contexts[k:k+transform_batch]
                X_sub = model.vectorizer.transform(sub_contexts, batch_size=len(sub_contexts), silent=True)
                X_chunks.append(X_sub)
            
            X_batch = np.vstack(X_chunks) if len(X_chunks) > 1 else X_chunks[0]
            del X_chunks
            
            # Train on this chunk
            if GPU_ACTIVE:
                X_batch_gpu = xp.asarray(X_batch, dtype=xp.float32)
                y_batch_gpu = xp.array(chunk_labels)
            else:
                X_batch_gpu = X_batch.astype(np.float32)
                y_batch_gpu = np.array(chunk_labels)
            
            # Forward pass
            logits = X_batch_gpu @ model.weights + model.bias
            max_logits = xp.max(logits, axis=1, keepdims=True)
            exp_logits = xp.exp(logits - max_logits)
            probs = exp_logits / xp.sum(exp_logits, axis=1, keepdims=True)
            
            # Loss
            loss = -xp.mean(xp.log(probs[xp.arange(len(chunk_labels)), y_batch_gpu] + 1e-10))
            loss += 0.5 * model.regularization * xp.sum(model.weights ** 2)
            epoch_loss += float(loss) if GPU_ACTIVE else loss
            
            # Gradients
            probs[xp.arange(len(chunk_labels)), y_batch_gpu] -= 1
            probs /= len(chunk_labels)
            grad_w = X_batch_gpu.T @ probs + model.regularization * model.weights
            grad_b = xp.sum(probs, axis=0)
            
            # Update
            model.weights -= model.learning_rate * grad_w
            model.bias -= model.learning_rate * grad_b
            
            batch_count += 1
            
            # Update progress bar with loss
            progress.set_postfix({'loss': f'{epoch_loss/(batch_count+1):.4f}'})
            
            # Clear memory aggressively
            del X_batch, X_batch_gpu, y_batch_gpu, chunk_contexts, chunk_labels
            del logits, probs, grad_w, grad_b
            gc.collect()
        
        print(f"✓ Epoch {epoch+1} complete, Avg Loss: {epoch_loss/batch_count:.4f}")
    
    print("\n✓ Training complete!")
    model.is_fitted = True
    
    # Build label mappings
    from src.config import DIACRITIC_TO_ID, ID_TO_DIACRITIC
    model.label_to_idx = DIACRITIC_TO_ID
    model.idx_to_label = ID_TO_DIACRITIC
    
    # Evaluate on dev set
    print("\n📊 Evaluating on dev set...")
    dev_contexts = []
    dev_labels_list = []
    for text, label_seq in zip(dev_texts, dev_labels):
        min_len = min(len(text), len(label_seq))
        for j in range(min_len):
            start = max(0, j - window_size // 2)
            end = min(len(text), j + window_size // 2 + 1)
            dev_contexts.append(text[start:end])
            dev_labels_list.append(label_seq[j])
    
    X_dev = model.vectorizer.transform(dev_contexts, batch_size=50000)
    
    # Predict
    try:
        import cupy as cp
        X_dev_gpu = cp.asarray(X_dev, dtype=cp.float32)
        logits = X_dev_gpu @ model.weights + model.bias
        preds = cp.argmax(logits, axis=1)
        preds_cpu = cp.asnumpy(preds)
    except:
        preds_cpu = np.argmax(X_dev @ model.weights.get() + model.bias.get(), axis=1)
    
    correct = np.sum(preds_cpu == np.array(dev_labels_list))
    total = len(dev_labels_list)
    accuracy = correct / total
    der = 1 - accuracy
    
    print("\n" + "="*70)
    print("✅ TRAINING COMPLETE")
    print("="*70)
    print(f"✓ Accuracy: {accuracy*100:.2f}%")
    print(f"✓ DER: {der*100:.2f}%")
    print(f"✓ Correct: {correct:,} / {total:,}")
    print("="*70)
    
    # Save model
    Path(MODEL_PATH).parent.mkdir(exist_ok=True, parents=True)
    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(model, f)
    print(f"✓ Model saved: {MODEL_PATH}")
